# Ray Serve

A comprehensive guide to Ray Serve for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

Ray Serve is a **scalable model serving library** built on top of Ray. It focuses on serving Python functions and ML models with support for **autoscaling, routing, and composition**.

### What is it?

- A serving layer that runs on a **Ray cluster**, managing deployments (replicas) of Python callables.  
- Exposes **HTTP and gRPC endpoints** for online inference.  
- Supports **multi-model composition**, request routing, and traffic splitting.

### Why use it?

Key benefits of using Ray Serve:

- **Unified serving for Python code and models** (not just ML models).  
- **Autoscaling and concurrency control** at the deployment level.  
- **Flexible composition** of deployments into applications.

### When to use it?

Ray Serve is particularly useful when:

- You already use **Ray** for training or data processing and want to reuse the same cluster for serving.  
- You need to serve multiple models or microservices that call each other.  
- You want fine-grained control over scaling and routing without managing a full separate serving system.

## Key Features

### Core Capabilities of Ray Serve

| Feature | Description | Benefit |
|--------|-------------|---------|
| **Deployments** | Python classes/functions decorated with `@serve.deployment`. | Simple abstraction for scalable services. |
| **Applications** | Groups of deployments composed into one app, via `serve.run` or config. | Build multi-step inference graphs. |
| **Autoscaling** | Scale replicas up/down based on load. | Efficient resource usage under variable traffic. |
| **HTTP/gRPC endpoints** | Serve requests over HTTP or gRPC. | Integrate with many clients and gateways. |
| **Model composition** | Deployments can call each other via handles. | Build ensembles and pipelines. |
| **Ray-native** | Uses Ray actors and object store underneath. | Leverages Ray’s distributed runtime. |

## Architecture Overview

Ray Serve runs on a Ray cluster and manages **deployment replicas** as Ray actors.

```text
+------------------------------+
|     Clients (HTTP/gRPC)      |
+---------------+--------------+
                |
                v
+------------------------------+
|     Ray Serve Proxy/Ingress  |
+---------------+--------------+
                |
                v
+------------------------------+
|     Ray Serve Controller     |
|  • Deployments & routes      |
|  • Autoscaling logic         |
+---------------+--------------+
                |
                v
+------------------------------+
|     Ray Cluster Workers      |
|  • Deployment replicas       |
+------------------------------+
```

You define deployments in Python, then run them on a Ray cluster (local or remote).

## Installation

### Prerequisites

- Python 3.8+.  
- Ray installed with Serve extras.

### Install Ray with Serve

```bash
pip install "ray[serve]"
```

For production, you’ll typically run Ray on a cluster (Kubernetes, VMs, etc.) and then deploy Serve applications there.

In [ ]:
# Quick helper: ensure Ray is available (uncomment to install)
# !pip install "ray[serve]"

try:
    import ray  # noqa: F401
    from ray import serve  # noqa: F401
    print("Ray and Ray Serve imports succeeded.")
except ImportError:
    print("Install with: pip install 'ray[serve]'")

## Basic Usage

### Minimal example: single deployment

Below is a simple Ray Serve deployment that echoes a message. This is a conceptual example; run it in a local environment where you can start Ray.

Steps:

1. Start Ray.  
2. Start Serve.  
3. Define and deploy a deployment.  
4. Send HTTP requests.

In [ ]:
# Minimal Ray Serve example (not executed here)

import ray
from ray import serve


@serve.deployment(num_replicas=1)
class Echo:
    async def __call__(self, request):
        data = await request.json()
        return {"echo": data}


if __name__ == "__main__":
    # Start Ray and Serve locally
    ray.init()
    serve.start()

    # Deploy the Echo service
    Echo.deploy()

    print("Echo deployment running at http://127.0.0.1:8000/Echo")
    # You can now send POST requests with JSON to that URL.

## Advanced Features

- **Composition of deployments**: Build graphs of deployments that call each other.  
- **Autoscaling policies**: Configure target concurrency and max replicas per deployment.  
- **Ingress via frameworks**: Integrate with FastAPI/Starlette using `@serve.ingress`.  
- **Deploy via config**: Use YAML application configs with `serve run` for production-like deployments.  
- **Traffic splitting**: Route a fraction of traffic to new versions for canary testing.

In [ ]:
# Sketch: composing deployments (conceptual)

from ray import serve


@serve.deployment
class ModelA:
    def __call__(self, data: str) -> str:
        return data.upper()


@serve.deployment
class ModelB:
    def __call__(self, data: str) -> str:
        return data[::-1]


@serve.deployment
class Ensemble:
    def __init__(self, model_a, model_b):
        self.model_a = model_a
        self.model_b = model_b

    async def __call__(self, request):
        text = await request.text()
        a = await self.model_a.remote(text)
        b = await self.model_b.remote(text)
        return {"a": a, "b": b}


# In a full script you would bind and run these deployments as an application.

## Use Cases

- **Online inference** for ML models (LLMs, vision models, recommendation models).  
- **Microservice-style Python applications** with autoscaling.  
- **Multi-model routers** that dispatch requests based on content or headers.  
- **Ensemble models** and multi-step inference pipelines running on Ray clusters.

## Best Practices

1. **Separate model loading from request handling**  
   - Initialize heavy models in `__init__` of the deployment, not on each request.

2. **Tune autoscaling parameters**  
   - Use realistic `max_concurrent_queries` and target concurrency per replica.

3. **Use async I/O when possible**  
   - For I/O-bound work (network calls, disk), use async methods to increase throughput.

4. **Manage environments carefully**  
   - Use Ray runtime environments or container images to ensure consistent dependencies across replicas.

5. **Use configuration files for production**  
   - Deploy Serve applications via YAML + `serve run` for reproducibility and CI/CD integration.

## Common Pitfalls

1. **Blocking operations in async handlers**  
   - Symptom: Low throughput despite high concurrency settings.  
   - Fix: Use async I/O or run blocking work in separate threads/processes.

2. **Underestimating model load time**  
   - Symptom: Slow scale-up due to heavy initialization.  
   - Fix: Pre-warm deployments, use lighter models for some traffic, or cache weights.

3. **Ignoring backpressure and timeouts**  
   - Symptom: Requests pile up, leading to high latencies.  
   - Fix: Configure timeouts and max concurrency, and scale out appropriately.

4. **Not monitoring Ray cluster health**  
   - Symptom: Serve issues that are actually Ray cluster resource problems.  
   - Fix: Monitor Ray dashboard and cluster metrics alongside Serve metrics.

## Performance Optimization

- **Profile latency and throughput** using Ray dashboard and application metrics.  
- **Right-size replicas**: choose appropriate CPU/GPU resources per deployment.  
- **Batching**: for some models, implement request batching inside deployments.  
- **Co-locate compute and data**: run Serve close to data sources or GPUs when possible.

Experiment with different replica counts, hardware types, and autoscaling configs to reach your SLOs.

In [ ]:
# Placeholder for performance benchmarking

print("Use load generators (e.g., locust, vegeta) and Ray dashboard to benchmark Serve deployments.")

## Production Deployment

- **Ray on Kubernetes**:  
  - Use Ray’s Kubernetes operator or Helm charts to run Ray clusters.  
  - Deploy Serve apps onto the cluster via configs or CI/CD pipelines.

- **Ingress & networking**:  
  - Front Serve with Ingress controllers, API gateways, or service meshes.  

- **Multi-tenant setups**:  
  - Use namespaces, Ray clusters per team, or logical app boundaries for isolation.

## Monitoring and Observability

- **Ray dashboard**:  
  - Inspect cluster resources, actor counts, and task throughput.  

- **Serve metrics**:  
  - Use built-in metrics for request latency, error rates, and replica status.  

- **Logging**:  
  - Centralize logs from Serve deployments and Ray workers for debugging.

Combine application-level metrics (e.g., P95 latency) with infrastructure metrics for full visibility.

## Troubleshooting

- **Serve app fails to start**:  
  - Check Python exceptions during deployment definition; inspect Ray logs.  

- **High error rates**:  
  - Inspect deployment logs; verify model loading, input validation, and error handling.

- **Scaling issues**:  
  - Confirm autoscaling config and cluster resource availability.

- **Cluster instability**:  
  - Investigate Ray cluster health and resource pressure; adjust node counts and resource limits.

## Comparison with Alternatives

| Aspect | Ray Serve | BentoML/OpenLLM | TGI / vLLM |
|--------|----------|-----------------|------------|
| Focus | General Python & ML serving on Ray | Model packaging & deployment | High-throughput LLM serving |
| Cluster runtime | Ray | Any (via containers) | Typically Kubernetes/VMs |
| Composition | Strong (deployment graphs) | Service composition via Bentos | Primarily single-model APIs |

Choose Ray Serve when you:

- Already use **Ray** or want a **general Python-native serving layer** with flexible composition and autoscaling.

## Resources

- Ray Serve docs: https://docs.ray.io/en/latest/serve/index.html  
- Key concepts: https://docs.ray.io/en/latest/serve/key-concepts.html  
- Deployment API: https://docs.ray.io/en/latest/serve/api/doc/ray.serve.Deployment.html  
- End-to-end tutorial: https://docs.ray.io/en/latest/serve/develop-and-deploy.html

These resources include detailed examples of building and deploying Ray Serve applications in production.